In [ ]:
# ============================================================
# TALTECH SEMANTIC PIPELINE
# NOTEBOOK 03 — BUILD NATIVE BRICK + SOSA RDF GRAPH
# ============================================================

from pathlib import Path
import hashlib
import pandas as pd

from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, XSD


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

EXTRACTION_DIR = OUTPUT_DIR / "01_extraction"
RECON_DIR = OUTPUT_DIR / "02_reconciliation"
OUT = OUTPUT_DIR / "03_rdf"

OUT.mkdir(
    parents=True,
    exist_ok=True
)

# Input files
OBS_FILE = (
    EXTRACTION_DIR /
    "02_selected_raw_observations_long.csv"
)

MAP_FILE = (
    RECON_DIR /
    "04_reconciliation_review_table.csv"
)

# Output RDF files
TTL_FILE = (
    OUT /
    "taltech_native_semantic_graph.ttl"
)

STATS_FILE = (
    OUT /
    "01_native_graph_statistics.csv"
)

print("Project directory :", PROJECT_DIR)
print("Observation file  :", OBS_FILE)
print("Reconciliation    :", MAP_FILE)
print("Output directory  :", OUT)


# ============================================================
# 2. INPUT VERIFICATION
# ============================================================

assert OBS_FILE.exists(), (
    "Observation file not found.\n"
    "Run Notebook 01 first:\n"
    f"{OBS_FILE}"
)

assert MAP_FILE.exists(), (
    "Reconciliation table not found.\n"
    "Run Notebook 02 first:\n"
    f"{MAP_FILE}"
)

print("\n✓ Observation file found")
print("✓ Reconciliation table found")


# ============================================================
# 3. LOAD OBSERVATIONS
# ============================================================

obs = pd.read_csv(
    OBS_FILE
)

obs["timestamp"] = pd.to_datetime(
    obs["timestamp"],
    utc=True,
    errors="coerce"
)

obs["value"] = pd.to_numeric(
    obs["value"],
    errors="coerce"
)

maps = pd.read_csv(
    MAP_FILE,
    dtype=str
).fillna("")


# ============================================================
# 4. VERIFY REQUIRED COLUMNS
# ============================================================

required_obs_columns = {
    "device_id",
    "timestamp",
    "variable",
    "value",
    "unit",
    "source_file"
}

missing_obs = (
    required_obs_columns
    - set(obs.columns)
)

assert not missing_obs, (
    "Missing observation columns: "
    f"{missing_obs}"
)

required_map_columns = {
    "operational_uuid",
    "ifc_global_id",
    "ifc_space_name",
    "mapping_origin",
    "mapping_status",
    "factual_location_claim",
    "mapping_confidence",
    "provenance",
    "approved_for_rdf"
}

missing_map = (
    required_map_columns
    - set(maps.columns)
)

assert not missing_map, (
    "Missing reconciliation columns: "
    f"{missing_map}"
)


# ============================================================
# 5. INPUT SUMMARY
# ============================================================

print("\n====================================")
print("INPUT SUMMARY")
print("====================================")

print(
    "Observation rows:",
    len(obs)
)

print(
    "Operational devices:",
    obs["device_id"].nunique()
)

print(
    "Variables:",
    sorted(
        obs["variable"]
        .dropna()
        .unique()
    )
)

print(
    "Reconciliation rows:",
    len(maps)
)


# ============================================================
# 6. RDF NAMESPACES
# ============================================================

EX = Namespace(
    "https://example.org/taltech-edt/"
)

BRICK = Namespace(
    "https://brickschema.org/schema/Brick#"
)

SOSA = Namespace(
    "http://www.w3.org/ns/sosa/"
)

PROV = Namespace(
    "http://www.w3.org/ns/prov#"
)

IFC = Namespace(
    "https://example.org/taltech-ifc/"
)


# ============================================================
# 7. INITIALIZE GRAPH
# ============================================================

g = Graph()

g.bind("ex", EX)
g.bind("brick", BRICK)
g.bind("sosa", SOSA)
g.bind("prov", PROV)
g.bind("ifc", IFC)

print("\n✓ Empty RDF graph initialized")


# ============================================================
# 8. SEMANTIC VARIABLE DEFINITIONS
# ============================================================

point_class = {

    "temperature":
        BRICK.Temperature_Sensor,

    "relative_humidity":
        BRICK.Relative_Humidity_Sensor,

    "co2":
        BRICK.CO2_Sensor
}

observed_property = {

    "temperature":
        EX.IndoorTemperature,

    "relative_humidity":
        EX.RelativeHumidity,

    "co2":
        EX.CO2Concentration
}

unsupported = (
    set(
        obs["variable"]
        .dropna()
        .unique()
    )
    -
    set(
        point_class.keys()
    )
)

assert not unsupported, (
    "Unsupported variables found: "
    f"{unsupported}"
)


# ============================================================
# 9. CREATE OPERATIONAL DEVICES
#    AND BRICK SEMANTIC POINTS
# ============================================================

devices = sorted(
    obs["device_id"]
    .dropna()
    .astype(str)
    .unique()
)

semantic_point_count = 0

for uuid in devices:

    device = EX[
        f"device/{uuid}"
    ]

    # Native operational device
    g.add(
        (
            device,
            RDF.type,
            EX.OperationalDevice
        )
    )

    # Preserve source UUID
    g.add(
        (
            device,
            EX.operationalUUID,
            Literal(uuid)
        )
    )

    device_variables = sorted(
        obs.loc[
            obs["device_id"].astype(str)
            == uuid,
            "variable"
        ]
        .dropna()
        .unique()
    )

    for variable in device_variables:

        point = EX[
            f"point/{uuid}/{variable}"
        ]

        # Brick semantic class
        g.add(
            (
                point,
                RDF.type,
                point_class[variable]
            )
        )

        # Semantic point linked to source device
        g.add(
            (
                point,
                EX.representsMeasurementFrom,
                device
            )
        )

        # Variable label
        g.add(
            (
                point,
                EX.variableName,
                Literal(variable)
            )
        )

        # Device has semantic point
        g.add(
            (
                device,
                EX.hasSemanticPoint,
                point
            )
        )

        semantic_point_count += 1


print("\n====================================")
print("STATIC NATIVE SEMANTIC STRUCTURE")
print("====================================")

print(
    "Operational devices:",
    len(devices)
)

print(
    "Semantic points:",
    semantic_point_count
)

print(
    "Static RDF triples:",
    len(g)
)


# ============================================================
# 10. CREATE SOSA OBSERVATIONS
# ============================================================

observation_count = 0
skipped_observations = 0

for idx, row in obs.iterrows():

    uuid = str(
        row["device_id"]
    )

    variable = str(
        row["variable"]
    )

    timestamp = row[
        "timestamp"
    ]

    value = row[
        "value"
    ]

    if (
        pd.isna(timestamp)
        or
        pd.isna(value)
    ):

        skipped_observations += 1
        continue

    point = EX[
        f"point/{uuid}/{variable}"
    ]

    device = EX[
        f"device/{uuid}"
    ]

    # Stable observation identifier
    key = (
        f"{uuid}|"
        f"{variable}|"
        f"{timestamp.isoformat()}|"
        f"{idx}"
    )

    digest = hashlib.sha1(
        key.encode("utf-8")
    ).hexdigest()

    observation = EX[
        f"observation/{digest}"
    ]

    # --------------------------------------------------------
    # SOSA observation type
    # --------------------------------------------------------

    g.add(
        (
            observation,
            RDF.type,
            SOSA.Observation
        )
    )

    # Timestamp
    g.add(
        (
            observation,
            SOSA.resultTime,
            Literal(
                timestamp.isoformat(),
                datatype=XSD.dateTime
            )
        )
    )

    # Numerical result
    g.add(
        (
            observation,
            SOSA.hasSimpleResult,
            Literal(
                float(value),
                datatype=XSD.double
            )
        )
    )

    # Observed property
    g.add(
        (
            observation,
            SOSA.observedProperty,
            observed_property[variable]
        )
    )

    # Observation → Brick semantic point
    g.add(
        (
            observation,
            EX.observedBySemanticPoint,
            point
        )
    )

    # Observation → operational device
    g.add(
        (
            observation,
            EX.sourceOperationalDevice,
            device
        )
    )

    # Original UUID
    g.add(
        (
            observation,
            EX.sourceOperationalUUID,
            Literal(uuid)
        )
    )

    # Original unit
    g.add(
        (
            observation,
            EX.sourceUnit,
            Literal(
                str(
                    row["unit"]
                )
            )
        )
    )

    # Source CSV provenance
    g.add(
        (
            observation,
            PROV.wasDerivedFrom,
            Literal(
                str(
                    row["source_file"]
                )
            )
        )
    )

    observation_count += 1


print("\n====================================")
print("SOSA OBSERVATIONS")
print("====================================")

print(
    "SOSA observations created:",
    observation_count
)

print(
    "Skipped observations:",
    skipped_observations
)

print(
    "RDF triples after observations:",
    len(g)
)


# ============================================================
# 11. CHECK RECONCILIATION STATUS
# ============================================================

approved = maps[
    maps["approved_for_rdf"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        [
            "true",
            "1",
            "yes"
        ]
    )
].copy()

print("\n====================================")
print("RECONCILIATION STATUS")
print("====================================")

print(
    "Reconciliation candidates:",
    len(maps)
)

print(
    "Approved mappings:",
    len(approved)
)

print(
    "Expected for G_native: 0"
)


# ============================================================
# 12. IMPORTANT:
#     BUILD NATIVE GRAPH ONLY
# ============================================================

# For the current experimental stage,
# NO spatial reconciliation mapping is added.
#
# This RDF graph represents G_native.
#
# The experimental reconciled graph G_reconciled
# will be generated later in a separate notebook/file.

if len(approved) > 0:

    print(
        "\nWARNING:"
    )

    print(
        "Some mappings are marked approved_for_rdf=True."
    )

    print(
        "They will NOT be inserted into G_native."
    )

    print(
        "G_native must remain free of experimental "
        "spatial mappings."
    )

native_mapping_count = 0


# ============================================================
# 13. SERIALIZE NATIVE RDF GRAPH
# ============================================================

g.serialize(
    destination=str(
        TTL_FILE
    ),
    format="turtle"
)

print("\n====================================")
print("NATIVE RDF GRAPH SAVED")
print("====================================")

print(
    "File:",
    TTL_FILE
)


# ============================================================
# 14. GRAPH STATISTICS
# ============================================================

rdf_observations = len(
    set(
        g.subjects(
            RDF.type,
            SOSA.Observation
        )
    )
)

rdf_devices = len(
    set(
        g.subjects(
            RDF.type,
            EX.OperationalDevice
        )
    )
)

rdf_points = len(
    set(
        g.subjects(
            EX.variableName,
            None
        )
    )
)

stats = pd.DataFrame(
    [
        {
            "metric":
                "graph_type",
            "value":
                "G_native"
        },
        {
            "metric":
                "rdf_triples",
            "value":
                len(g)
        },
        {
            "metric":
                "operational_devices",
            "value":
                rdf_devices
        },
        {
            "metric":
                "semantic_points",
            "value":
                rdf_points
        },
        {
            "metric":
                "sosa_observations",
            "value":
                rdf_observations
        },
        {
            "metric":
                "source_observation_rows",
            "value":
                len(obs)
        },
        {
            "metric":
                "approved_spatial_mappings_used",
            "value":
                native_mapping_count
        }
    ]
)

stats.to_csv(
    STATS_FILE,
    index=False
)

print("\n====================================")
print("NATIVE GRAPH STATISTICS")
print("====================================")

print(
    stats.to_string(
        index=False
    )
)


# ============================================================
# 15. TRACEABILITY CHECKS
# ============================================================

print("\n====================================")
print("TRACEABILITY CHECKS")
print("====================================")

print(
    "Source rows       :",
    len(obs)
)

print(
    "RDF observations  :",
    rdf_observations
)

print(
    "Devices           :",
    rdf_devices
)

print(
    "Semantic points   :",
    rdf_points
)

print(
    "Spatial mappings  :",
    native_mapping_count
)


# ============================================================
# 16. CONSISTENCY ASSERTIONS
# ============================================================

assert rdf_observations == observation_count, (
    "Mismatch between generated SOSA "
    "observations and RDF observations."
)

assert rdf_devices == len(devices), (
    "Mismatch in operational device count."
)

assert rdf_points == semantic_point_count, (
    "Mismatch in Brick semantic point count."
)

assert native_mapping_count == 0, (
    "G_native must contain zero spatial mappings."
)

print(
    "\n✓ Observation preservation verified"
)

print(
    "✓ Operational device count verified"
)

print(
    "✓ Semantic point count verified"
)

print(
    "✓ No spatial reconciliation inserted"
)

print(
    "✓ G_native successfully created"
)


# ============================================================
# 17. OUTPUT FILES
# ============================================================

print("\n====================================")
print("FILES CREATED")
print("====================================")

for output_file in sorted(
    OUT.iterdir()
):

    print(
        " -",
        output_file.name
    )